In [1]:
# descargamos los datos del ONI indicador principal de la NOAA (Oficina Nacional de Administración Oceánica y 
# Atmosférica (National Oceanic and Atmospheric Administration). Se trata de una prestigiosa agencia científica del 
# gobierno de los Estados Unidos —dependiente delDepartamento de Comercio— encargada de monitorear y estudiar las 
# condiciones del océano, la atmósfera y el espacio)
# para medir y clasificar las fases de El Niño-Oscilación del Sur (ENOS) 
# en el Océano Pacífico ecuatorial
import urllib.request

url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
urllib.request.urlretrieve(url, "oni_historico.txt")
print("Descargado correctamente")


Descargado correctamente


In [2]:
with open("oni_historico.txt") as f:
    for i in range(10):
        print(repr(f.readline()))

' SEAS  YR   TOTAL   ANOM\n'
'  DJF 1950  25.01  -1.32\n'
'  JFM 1950  25.36  -1.20\n'
'  FMA 1950  25.88  -1.12\n'
'  MAM 1950  26.24  -1.08\n'
'  AMJ 1950  26.35  -1.10\n'
'  MJJ 1950  26.32  -0.90\n'
'  JJA 1950  26.24  -0.62\n'
'  JAS 1950  25.94  -0.57\n'
'  ASO 1950  25.76  -0.56\n'


In [3]:
# SEAS -> ESTACIÓN DEL AÑO
# ANOM -> valor ONI 
# verano ( DJF)
# invierno ( JJA)
# comparamos el ONI de DJF de un año con las lluvias de JJA de ese mismo año.


In [4]:
# cargamos y filtramos el ONI de verano ( DJF)
import pandas as pd

df_oni = pd.read_csv("oni_historico.txt", sep=r"\s+")
df_oni.columns = ["temporada", "year", "total", "anom"]

oni_verano = df_oni[df_oni["temporada"] == "DJF"][["year", "anom"]].rename(columns={"anom": "oni_verano"})
print(f"Total de veranos con dato ONI: {len(oni_verano)}")
oni_verano.head()

Total de veranos con dato ONI: 77


,year,oni_verano
0,1950,-1.32
12,1951,-0.66
24,1952,0.52
36,1953,0.25
48,1954,0.41


In [5]:
df_precip = pd.read_excel("EC_series.xlsx")
df_precip.head()

,agno,mes,dia,valor
0,1974,1,1,0.0
1,1974,1,2,0.0
2,1974,1,3,0.0
3,1974,1,4,0.0
4,1974,1,5,0.0


In [6]:
df_precip = pd.read_excel("EC_series.xlsx")
print(df_precip.columns.tolist())
df_precip.head()

['agno', 'mes', 'dia', 'valor']


,agno,mes,dia,valor
0,1974,1,1,0.0
1,1974,1,2,0.0
2,1974,1,3,0.0
3,1974,1,4,0.0
4,1974,1,5,0.0


In [7]:
# Sumamos la precipitación de invierno (junio, julio, agosto) por año
precip_invierno = df_precip[df_precip["mes"].isin([6, 7, 8])].groupby("agno")["valor"].sum().reset_index()
precip_invierno = precip_invierno.rename(columns={"agno": "year", "valor": "precip_invierno_mm"})

print(f"Total de inviernos con datos: {len(precip_invierno)}")
precip_invierno.head()

Total de inviernos con datos: 46


,year,precip_invierno_mm
0,1974,307.0
1,1975,363.0
2,1976,90.0
3,1977,546.0
4,1978,410.0


In [8]:
dataset_nino = pd.merge(oni_verano, precip_invierno, on="year")
print(f"Años con ambos datos (ONI verano + lluvia invierno): {len(dataset_nino)}")

correlacion = dataset_nino[["oni_verano", "precip_invierno_mm"]].corr()
print(correlacion)

Años con ambos datos (ONI verano + lluvia invierno): 46
                    oni_verano  precip_invierno_mm
oni_verano            1.000000           -0.033738
precip_invierno_mm   -0.033738            1.000000


In [9]:
# correlación casi inexistente -0.03


In [10]:
# Contamos cuántos días con dato tiene cada invierno (debería ser ~92 días: jun+jul+ago)
dias_por_invierno = df_precip[df_precip["mes"].isin([6,7,8])].groupby("agno")["valor"].count().reset_index()
dias_por_invierno = dias_por_invierno.rename(columns={"agno":"year","valor":"dias_con_dato"})
dias_por_invierno.sort_values("dias_con_dato").head(10)

,year,dias_con_dato
0,1974,90
2,1976,90
3,1977,90
1,1975,91
9,1983,91
33,2007,91
43,2017,91
4,1978,92
7,1981,92
8,1982,92


In [11]:

# descartado que fuera un problema de datos faltantes.
# El siguiente paso es ver si la relación es más "categórica" que lineal
# comparando el promedio de lluvia según si el año fue Niño, Niña o Neutro:
def clasificar_enso(oni):
    if oni >= 0.5:
        return "Niño"
    elif oni <= -0.5:
        return "Niña"
    else:
        return "Neutro"

dataset_nino["categoria_enso"] = dataset_nino["oni_verano"].apply(clasificar_enso)

resumen = dataset_nino.groupby("categoria_enso")["precip_invierno_mm"].agg(["mean", "median", "count"])
print(resumen)

                      mean  median  count
categoria_enso                           
Neutro          324.435714  296.90     14
Niña            309.443750  280.35     16
Niño            322.150000  312.75     16


In [12]:
#Los promedios sí muestran una tendencia en la dirección esperada (Niño: 322mm > Niña: 309mm), pero la diferencia 
#es pequeña comparada con la variabilidad dentro 
#de cada grupo — por eso la correlación general salió casi plana. Antes de descartar la relación, hay dos cosas
#rigurosas que nos faltan hacer.

#1. Probar si la diferencia es estadísticamente significativa (no solo mirar el promedio a ojo — con solo 
#16 años por grupo, una diferencia de 13mm bien podría ser ruido):
from scipy import stats

lluvia_nino = dataset_nino[dataset_nino["categoria_enso"] == "Niño"]["precip_invierno_mm"]
lluvia_nina = dataset_nino[dataset_nino["categoria_enso"] == "Niña"]["precip_invierno_mm"]

t_stat, p_valor = stats.ttest_ind(lluvia_nino, lluvia_nina)
print(f"Valor p: {p_valor:.3f}")

Valor p: 0.806


In [14]:
# Cómo interpretarlo: si p_valor es menor a 0.05, la diferencia es estadísticamente significativa 
# (poco probable que sea solo azar). Si es mayor, no podemos afirmar con confianza que Niño y Niña sean
# realmente distintos con 
# estos datos.
#Un p de 0.806 es muy alto — estadísticamente, no hay evidencia de diferencia real entre años Niño y Niña en esta 
#estación específica. Toca ser honesto: con los datos actuales, la relación esperada no se confirma en Pichidegua.

#Antes de aceptar esto como conclusión final, hay dos causas más por descartar — y son metodológicamente
#importantes, no solo "intentar hasta que funcione":

#1. Ventana de lluvia demasiado estrecha. JJA (jun-ago) es el invierno astronómico, pero en Chile central la
#temporada de lluvias relevante suele ser más amplia (abril-septiembre). Si cortamos la temporada muy justo,
#podríamos estar perdiendo parte de la señal real:

In [15]:
precip_lluvias = df_precip[df_precip["mes"].isin([4,5,6,7,8,9])].groupby("agno")["valor"].sum().reset_index()
precip_lluvias = precip_lluvias.rename(columns={"agno": "year", "valor": "precip_lluvias_mm"})

dataset_nino2 = pd.merge(oni_verano, precip_lluvias, on="year")
print(dataset_nino2[["oni_verano", "precip_lluvias_mm"]].corr())

                   oni_verano  precip_lluvias_mm
oni_verano           1.000000          -0.051811
precip_lluvias_mm   -0.051811           1.000000


In [ ]:
# El desfase que usamos probablemente no es el correcto. Los análisis reales usan el ONI concurrente con la
# temporada de lluvias (mayo-agosto, la misma ventana que la lluvia), no el ONI del verano previo. Es decir: no es
# verano anticipa invierno", sino "el estado del Pacífico durante el invierno mismo" el que se correlaciona mejor
# tiene sentido físico, porque la teleconexión atmosférica opera mientras el fenómeno está activo, no como un
# pronóstico adelantado.

In [ ]:
# según un análisis reciente del CR2 (uno de los centros de investigación climática más serios de Chile), la
# correlación entre El Niño y la lluvia en Chile central se debilitó fuertemente después del año 2000 de sobre
# 0.7 entre 1970-1990, a solo ~0.2 después de 2000 (por causas aún no bien entendidas, asociadas a la "megasequía" 
# que afecta a Chile central desde hace más de una década). Como tus datos van de 1974 a 2020, estás promediando una
# época de relación fuerte con otra de relación débil/rota eso puede estar aplanando tu correlación general.

In [16]:
# 1. Usamos el ONI promedio de mayo-agosto (concurrente), no el de verano
oni_mjja = df_oni[df_oni["temporada"].isin(["MJJ","JJA"])].groupby("year")["anom"].mean().reset_index()
oni_mjja = oni_mjja.rename(columns={"anom": "oni_mjja"})

dataset_v2 = pd.merge(oni_mjja, precip_invierno, on="year")

# 2. Dividimos en dos períodos para ver si la relación se debilitó, igual que documenta la literatura
periodo1 = dataset_v2[dataset_v2["year"] < 2000]
periodo2 = dataset_v2[dataset_v2["year"] >= 2000]

print("1974-1999:", periodo1[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])
print("2000-2020:", periodo2[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])
print("Todo el período:", dataset_v2[["oni_mjja","precip_invierno_mm"]].corr().iloc[0,1])

1974-1999: 0.46860216181871667
2000-2020: -0.027616965013425466
Todo el período: 0.32178985549403105


In [ ]:
## 1974-1999: correlación de 0.47 (moderada-fuerte)  la relación El Niño-lluvia funcionaba bien
## 2000-2020: correlación de -0.03 (nula)  la relación se rompió
## Todo el período junto: 0.32  un promedio engañoso que esconde ambos comportamientos

In [ ]:
# el resultado es rico porque cuenta una historia real: algo cambió en el sistema climático
# no que la relación nunca existió.

In [ ]:
# el desfase que asumimos al principio (verano→invierno) estaba mal — la relación correcta es concurrente
# (MJJA↔MJJA), y eso también fue parte del aprendizaje: revisar la literatura antes de asumir un desfase
# temporal "porque suena lógico".

In [ ]:
# Si el RONI mejora la correlación del período reciente, eso sería un hallazgo muy fuerte para tu paper: 
# "el debilitamiento observado con ONI es parcialmente un artefacto de la definición del índice, no una desaparición
# real de la teleconexión". Si no mejora, entonces el fenómeno es más profundo (cambios en la circulación atmosférica,
#" como sugiere la literatura) y probamos la siguiente hipótesis: un índice distinto como el AAO (Oscilación Antártica)
# que varios estudios chilenos mencionan como cada vez más relevante para el clima de Chile centro-sur.

In [17]:
import urllib.request

url_roni = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"
urllib.request.urlretrieve(url_roni, "roni_historico.txt")

df_roni = pd.read_csv("roni_historico.txt", sep=r"\s+")
df_roni.columns = ["temporada", "year", "anom_roni"]

roni_mjja = df_roni[df_roni["temporada"].isin(["MJJ","JJA"])].groupby("year")["anom_roni"].mean().reset_index()
roni_mjja = roni_mjja.rename(columns={"anom_roni": "roni_mjja"})

dataset_roni = pd.merge(roni_mjja, precip_invierno, on="year")

periodo2_roni = dataset_roni[dataset_roni["year"] >= 2000]
print("2000-2020 con RONI:", periodo2_roni[["roni_mjja","precip_invierno_mm"]].corr().iloc[0,1])
print("(para comparar, con ONI era: -0.03)")

2000-2020 con RONI: 0.1691966770726637
(para comparar, con ONI era: -0.03)


In [ ]:
# Mejora, pero no resuelve el misterio del todo — de -0.03 a 0.17 es una mejora real, indicando que parte del
# problema sí era el sesgo del ONI tradicional, pero 0.17 sigue siendo una relación débil.

In [18]:
url_aao = "https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/aao/monthly.aao.index.b79.current.ascii.table"
urllib.request.urlretrieve(url_aao, "aao_historico.txt")

df_aao = pd.read_csv("aao_historico.txt", sep=r"\s+")
df_aao = df_aao.rename(columns={df_aao.columns[0]: "year"})
df_aao.head()

,year,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
1979,0.209,0.356,0.899,0.678,0.724,1.700,2.412,0.546,0.629,0.160,-0.423,-0.951
1980,-0.447,-0.980,-1.424,-2.068,-0.479,0.286,-1.944,-0.997,-1.701,0.577,-2.013,-0.356
1981,0.231,0.039,-0.966,-1.462,-0.344,0.352,-0.986,-2.118,-1.509,-0.260,0.626,1.116
1982,-0.554,0.277,1.603,1.531,0.118,0.920,-0.415,0.779,1.580,-0.702,-0.849,-1.934
1983,-1.340,-1.081,0.166,0.149,-0.437,-0.263,1.114,0.792,-0.696,1.193,0.727,0.475


In [19]:
print(df_aao.columns.tolist())
df_aao.head()

['year', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


,year,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
1979,0.209,0.356,0.899,0.678,0.724,1.700,2.412,0.546,0.629,0.160,-0.423,-0.951
1980,-0.447,-0.980,-1.424,-2.068,-0.479,0.286,-1.944,-0.997,-1.701,0.577,-2.013,-0.356
1981,0.231,0.039,-0.966,-1.462,-0.344,0.352,-0.986,-2.118,-1.509,-0.260,0.626,1.116
1982,-0.554,0.277,1.603,1.531,0.118,0.920,-0.415,0.779,1.580,-0.702,-0.849,-1.934
1983,-1.340,-1.081,0.166,0.149,-0.437,-0.263,1.114,0.792,-0.696,1.193,0.727,0.475


In [20]:
df_aao["aao_mjja"] = df_aao[["May","Jun","Jul","Aug"]].mean(axis=1)
aao_mjja = df_aao[["year","aao_mjja"]]

dataset_aao = pd.merge(aao_mjja, precip_invierno, on="year")

periodo2_aao = dataset_aao[dataset_aao["year"] >= 2000]
print("2000-2020, AAO vs lluvia:", periodo2_aao[["aao_mjja","precip_invierno_mm"]].corr().iloc[0,1])

2000-2020, AAO vs lluvia: nan


C:\Users\haltv\AppData\Local\Temp\ipykernel_8708\3717215053.py:4: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  dataset_aao = pd.merge(aao_mjja, precip_invierno, on="year")


In [21]:
df_aao = pd.read_csv("aao_historico.txt", sep=r"\s+", skiprows=1, header=None,
                      names=["year","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
df_aao["year"] = df_aao["year"].astype(int)

df_aao["aao_mjja"] = df_aao[["May","Jun","Jul","Aug"]].mean(axis=1)
aao_mjja = df_aao[["year","aao_mjja"]]

dataset_aao = pd.merge(aao_mjja, precip_invierno, on="year")
print(f"Años cruzados: {len(dataset_aao)}")

periodo2_aao = dataset_aao[dataset_aao["year"] >= 2000]
print("2000-2020, AAO vs lluvia:", periodo2_aao[["aao_mjja","precip_invierno_mm"]].corr().iloc[0,1])

Años cruzados: 41
2000-2020, AAO vs lluvia: -0.32311549482450413


In [ ]:
# el signo negativo tiene lógica atmosférica clara — cuando el AAO está en fase positiva (los vientos del
# oeste/tormentas se contraen hacia el polo, alejándose de Chile central), el resultado es menos lluvia. Y aquí 
# viene lo interesante: el AAO ha mostrado una tendencia positiva sostenida en las últimas décadas (asociada al 
                                                                                           
# calentamiento global y el agujero de ozono histórico) — lo que coincide exactamente con el período de la megasequía.
  #  Esto sugiere una historia coherente: no es que El Niño dejó de importar, sino que el AAO se volvió un factor 
# competidor cada vez más dominante, empujando hacia menos lluvia incluso en años Niño.

In [22]:
dataset_completo = pd.merge(roni_mjja, aao_mjja, on="year")
dataset_completo = pd.merge(dataset_completo, precip_invierno, on="year")
dataset_completo_2000 = dataset_completo[dataset_completo["year"] >= 2000]

print(dataset_completo_2000[["roni_mjja","aao_mjja","precip_invierno_mm"]].corr())

                    roni_mjja  aao_mjja  precip_invierno_mm
roni_mjja            1.000000 -0.158305            0.169197
aao_mjja            -0.158305  1.000000           -0.323115
precip_invierno_mm   0.169197 -0.323115            1.000000


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_combo = dataset_completo_2000[["roni_mjja", "aao_mjja"]]
y_combo = dataset_completo_2000["precip_invierno_mm"]

modelo_combo = LinearRegression()
modelo_combo.fit(X_combo, y_combo)
pred_combo = modelo_combo.predict(X_combo)

print(f"R² combinado (RONI + AAO): {r2_score(y_combo, pred_combo):.3f}")
print(f"R² solo RONI: {0.169**2:.3f}")
print(f"R² solo AAO: {0.323**2:.3f}")

R² combinado (RONI + AAO): 0.119
R² solo RONI: 0.029
R² solo AAO: 0.104


In [ ]:
# El modelo combinado (R²=0.119) explica solo ~12% de la variabilidad de la lluvia reciente, y curiosamente es
# incluso un poco menor que la suma simple de ambos efectos individuales (0.029+0.104=0.133) — esto sugiere que 
# ambos índices comparten algo de información redundante entre sí, o que simplemente hay mucho ruido con solo 21 
# datos.
# El AAO es, de los tres índices probados (ONI, RONI, AAO), el que mejor explica individualmente la lluvia de
# invierno en el período reciente (2000-2020) — consistente con la literatura que apunta a un rol creciente del 
# AAO en el debilitamiento de la señal de El Niño.
# Pero incluso el mejor predictor solo explica ~10% de la variabilidad — la mayoría de la variación de la lluvia 
# en este período sigue sin explicarse por estos índices climáticos grandes, lo cual también es información real: 
# con una sola estación y ~20 años de datos, hay bastante ruido local que estos índices globales no capturan.

In [ ]:
# el AAO parece jugar un rol más importante que ONI en años recientes, pero la relación general es débil y con 
# alta incertidumbre dado el tamaño muestral

In [25]:
import urllib.request
import zipfile

url_cr2 = "https://www.cr2.cl/download/cr2_pramon_2019-zip/?wpdmdl=25416"

req = urllib.request.Request(url_cr2, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as response, open("cr2_prAmon_2019.zip", "wb") as f:
    f.write(response.read())

with zipfile.ZipFile("cr2_prAmon_2019.zip", "r") as z:
    print(z.namelist())
    z.extractall("cr2_precipitacion")

print("Descargado y descomprimido correctamente")

['cr2_prAmon_2019/', 'cr2_prAmon_2019/cr2_prAmon_2019_stations.txt', 'cr2_prAmon_2019/cr2_prAmon_2019.txt', 'cr2_prAmon_2019/cr2_prAmon_2019_description.txt', 'cr2_prAmon_2019/cr2_prAmon_2019.html']
Descargado y descomprimido correctamente


In [26]:
df_estaciones = pd.read_csv("cr2_precipitacion/cr2_prAmon_2019/cr2_prAmon_2019_stations.txt")
print(df_estaciones.columns.tolist())
df_estaciones.head()

['codigo_estacion', 'institucion', 'fuente', 'nombre', 'altura', 'latitud', 'longitud', 'codigo_cuenca', 'nombre_cuenca', 'codigo_sub_cuenca', 'nombre_sub_cuenca', 'inicio_observaciones', 'fin_observaciones', 'cantidad_observaciones', 'inicio_automatica']


,codigo_estacion,institucion,fuente,nombre,altura,latitud,longitud,codigo_cuenca,nombre_cuenca,codigo_sub_cuenca,nombre_sub_cuenca,inicio_observaciones,fin_observaciones,cantidad_observaciones,inicio_automatica
0,1000005,DGA,dga_web,Visviri,4080,-17.5950,-69.4831,10,Altiplanicas,100,Entre Limite Peru-Bolivia Y Rio Lauca,1968-05-01,2019-12-31,18167,2017-08-02
1,1200002,DGA,dga_web,Villa Industrial (Tacora),4080,-17.7719,-69.7244,12,Rio Lluta,120,Rio Lluta Alto,1975-01-01,2019-11-30,15794,-
2,1200003,DGA,dga_web,Humapalca,3980,-17.8350,-69.7039,12,Rio Lluta,120,Rio Lluta Alto,1971-12-01,2019-11-30,17393,-
3,1201005,DGA,dga_web,Rio Caracarani En Humapalca,3908,-17.8428,-69.6994,12,Rio Lluta,120,Rio Lluta Alto,2013-06-01,2019-12-31,2185,2017-04-04
4,1201010,DGA,dga_web,Alcerreca,3990,-17.9931,-69.6594,12,Rio Lluta,120,Rio Lluta Alto,1971-01-01,2019-11-30,17563,-


In [27]:
df_ohiggins = df_estaciones[df_estaciones["codigo_estacion"].astype(str).str.startswith("06")]

# Nos interesan las que tengan buena cantidad de datos y cubran gran parte de 1974-2020
df_ohiggins_buenas = df_ohiggins[df_ohiggins["cantidad_observaciones"] > 300].sort_values("cantidad_observaciones", ascending=False)

print(f"Estaciones en O'Higgins con buen historial: {len(df_ohiggins_buenas)}")
df_ohiggins_buenas[["codigo_estacion","nombre","inicio_observaciones","fin_observaciones","cantidad_observaciones"]].head(15)

Estaciones en O'Higgins con buen historial: 0


,codigo_estacion,nombre,inicio_observaciones,fin_observaciones,cantidad_observaciones


In [28]:
codigos_str = df_estaciones["codigo_estacion"].astype(str).str.zfill(8)
df_ohiggins = df_estaciones[codigos_str.str.startswith("06")]

df_ohiggins_buenas = df_ohiggins[df_ohiggins["cantidad_observaciones"] > 300].sort_values("cantidad_observaciones", ascending=False)

print(f"Estaciones en O'Higgins con buen historial: {len(df_ohiggins_buenas)}")
df_ohiggins_buenas[["codigo_estacion","nombre","inicio_observaciones","fin_observaciones","cantidad_observaciones"]].head(15)

Estaciones en O'Higgins con buen historial: 39


,codigo_estacion,nombre,inicio_observaciones,fin_observaciones,cantidad_observaciones
427,6027003,La Rufina,1929-05-01,2019-09-30,32772
393,6056003,Rapel,1940-07-01,2019-09-30,28474
414,6013005,Popeta,1970-01-01,2019-09-30,17894
422,6016004,San Fernando,1971-09-01,2019-12-31,17623
421,6132002,Nilahue Barahona,1969-01-01,2019-08-31,17602
429,6034003,Convento Viejo,1971-09-01,2019-09-30,17525
413,6015003,Rengo,1971-01-01,2019-09-30,17304
418,6018010,Millahue,1972-01-01,2019-09-30,16662
407,6019005,Pichidegua,1974-01-01,2019-09-30,16484
430,6036001,La Candelaria,1974-05-01,2019-09-30,16356


In [30]:
with open("cr2_precipitacion/cr2_prAmon_2019/cr2_prAmon_2019.txt", encoding="latin-1") as f:
    for i in range(8):
        print(f.readline()[:150])

codigo_estacion,1000005,1200002,1200003,1201005,1201010,1201003,1001005,1110001,1202009,1020014,1020013,1020002,1202012,1202010,1020015,1020017,120201
institucion,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DGA,DG
fuente,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web,dga_web
nombre,Visviri,Villa Industrial (Tacora),Humapalca,Rio Caracarani En Humapalca,Alcerreca,Rio Lluta En Alcerreca,Caquena,Puquios,Pacollo,Cotacotani,Isl
altura,4080,4080,3980,3908,3990,3550,4400,3750,4185,4550,4540,4500,3560,3545,4420,4420,4060,4400,4400,4585,4600,4570,4375,4720,3550,3350,290,3240,20,5
latitud,-17.595,-17.7719,-17.835,-17.8428,-17.9931,-18.0036,-18.0542,-18.1747,-18.1769,-18.1836,-18.1858,-18.1936,-18.195,-18.1992,-18.2014,-18.2042,-
longitud,-69.4831,-69.7244,-69.7039,-69.6994,-69.6594,-69.6331,-69.2017,-69.7439,-69.5092,-69.

In [31]:
df_series = pd.read_csv("cr2_precipitacion/cr2_prAmon_2019/cr2_prAmon_2019.txt", 
                          encoding="latin-1", skiprows=14)
df_series = df_series.rename(columns={df_series.columns[0]: "fecha"})

codigos_elegidos = ["6019005", "6016004", "6015003", "6010015", "6034003"]  # Pichidegua, San Fernando, Rengo, Rancagua, Convento Viejo

# Verificamos cuáles de estos códigos existen como columna
disponibles = [c for c in codigos_elegidos if c in df_series.columns]
print("Estaciones encontradas como columnas:", disponibles)
df_series[["fecha"] + disponibles].head()

Estaciones encontradas como columnas: []


,fecha
0,1900-01
1,1900-02
2,1900-03
3,1900-04
4,1900-05


In [32]:
print(df_series.columns.tolist()[:10])

['fecha', '2017-08-02', '-', '-.1', '2017-04-04', '-.2', '2017-09-15', '-.3', '-.4', '-.5']


In [33]:
with open("cr2_precipitacion/cr2_prAmon_2019/cr2_prAmon_2019.txt", encoding="latin-1") as f:
    lineas = f.readlines()

fila_header = next(i for i, l in enumerate(lineas) if l.startswith("codigo_estacion"))
print(f"La fila de encabezado real está en la línea: {fila_header}")

df_series = pd.read_csv("cr2_precipitacion/cr2_prAmon_2019/cr2_prAmon_2019.txt",
                          encoding="latin-1", skiprows=fila_header)
df_series = df_series.rename(columns={df_series.columns[0]: "fecha"})

codigos_elegidos = ["6019005", "6016004", "6015003", "6010015", "6034003"]
disponibles = [c for c in codigos_elegidos if c in df_series.columns]
print("Estaciones encontradas como columnas:", disponibles)
df_series[["fecha"] + disponibles].head()

La fila de encabezado real está en la línea: 0
Estaciones encontradas como columnas: ['6019005', '6016004', '6015003', '6010015', '6034003']


C:\Users\haltv\AppData\Local\Temp\ipykernel_8708\3148678231.py:7: DtypeWarning: Columns (0: 1000005, 1: 1200002, 2: 1200003, 3: 1201005, 4: 1201010, 5: 1201003, 6: 1001005, 7: 1110001, 8: 1202009, 9: 1020014, 10: 1020013, 11: 1020002, 12: 1202012, 13: 1202010, 14: 1020015, 15: 1020017, 16: 1202011, 17: 1020016, 18: 1020018, 19: 1010010, 20: 1010009, 21: 1010007, 22: 1300005, 23: 1010008, 24: 1300004, 25: 1300006, 26: 1211006, 27: 1300007, 28: 1310018, 29: 1310021, 30: 1021002, 31: 1021007, 32: 1310022, 33: 1300009, 34: 1310019, 35: 1300008, 36: 1021001, 37: 1030003, 38: 1410011, 39: 1410012, 40: 1502007, 41: 1502008, 42: 1501001, 43: 1610003, 44: 1502006, 45: 1610004, 46: 1041004, 47: 1041003, 48: 1611002, 49: 1611001, 50: 1720004, 51: 1720006, 52: 1730020, 53: 1730015, 54: 1730007, 55: 1730019, 56: 1730016, 57: 1042001, 58: 1044001, 59: 1730017, 60: 1730018, 61: 1042002, 62: 1740001, 63: 1050007, 64: 1740002, 65: 1750003, 66: 1050009, 67: 1050004, 68: 1700010, 69: 1051004, 70: 1750002

,fecha,6019005,6016004,6015003,6010015,6034003
0,institucion,DGA,DGA,DGA,DGA,DGA
1,fuente,dga_web,dga_web,dga_web,dga_web,dga_web
2,nombre,Pichidegua,San Fernando,Rengo,Rancagua (Cachapoal - Dcp),Convento Viejo
3,altura,110,350,310,515,239
4,latitud,-34.2872,-34.5983,-34.4217,-34.1908,-34.7694


In [34]:
# Buscamos dónde empiezan realmente los datos (después de todas las filas de metadatos)
fila_datos_inicio = df_series[df_series["fecha"].astype(str).str.match(r"^\d{4}-\d{2}$")].index[0]
df_series_limpio = df_series.iloc[fila_datos_inicio:].copy()

df_series_limpio = df_series_limpio.rename(columns={
    "6019005": "pichidegua", "6016004": "san_fernando", "6015003": "rengo",
    "6010015": "rancagua", "6034003": "convento_viejo"
})

df_series_limpio[["fecha","pichidegua","san_fernando","rengo","rancagua","convento_viejo"]] = \
    df_series_limpio[["fecha","pichidegua","san_fernando","rengo","rancagua","convento_viejo"]].apply(pd.to_numeric, errors="coerce")

print(df_series_limpio.shape)
df_series_limpio[["fecha","pichidegua","san_fernando","rengo","rancagua","convento_viejo"]].head()

(1440, 880)


,fecha,pichidegua,san_fernando,rengo,rancagua,convento_viejo
14,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
15,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
16,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
17,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
18,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0


In [35]:
df_series_limpio = df_series.iloc[fila_datos_inicio:].copy()
df_series_limpio = df_series_limpio.rename(columns={
    "6019005": "pichidegua", "6016004": "san_fernando", "6015003": "rengo",
    "6010015": "rancagua", "6034003": "convento_viejo"
})

columnas_estaciones = ["pichidegua","san_fernando","rengo","rancagua","convento_viejo"]

# Solo convertimos a número las columnas de estaciones, NO la fecha
df_series_limpio[columnas_estaciones] = df_series_limpio[columnas_estaciones].apply(pd.to_numeric, errors="coerce")

# Reemplazamos el valor centinela -9999 por NaN real
df_series_limpio[columnas_estaciones] = df_series_limpio[columnas_estaciones].replace(-9999.0, pd.NA)

print(df_series_limpio.shape)
df_series_limpio[["fecha"] + columnas_estaciones].head()

(1440, 880)


,fecha,pichidegua,san_fernando,rengo,rancagua,convento_viejo
14,1900-01,<NA>,<NA>,<NA>,<NA>,<NA>
15,1900-02,<NA>,<NA>,<NA>,<NA>,<NA>
16,1900-03,<NA>,<NA>,<NA>,<NA>,<NA>
17,1900-04,<NA>,<NA>,<NA>,<NA>,<NA>
18,1900-05,<NA>,<NA>,<NA>,<NA>,<NA>


In [36]:
df_series_limpio["year"] = df_series_limpio["fecha"].str[:4].astype(int)
df_series_limpio["month"] = df_series_limpio["fecha"].str[5:7].astype(int)

invierno = df_series_limpio[df_series_limpio["month"].isin([6,7,8])]
invierno_anual = invierno.groupby("year")[columnas_estaciones].sum(min_count=3)  # exige que existan los 3 meses

# Promedio regional: el promedio de las 5 estaciones para cada año
invierno_anual["precip_regional_mm"] = invierno_anual[columnas_estaciones].mean(axis=1)

precip_regional = invierno_anual[["precip_regional_mm"]].reset_index()
print(f"Años con datos: {len(precip_regional)}")
precip_regional.tail(10)

Años con datos: 120


,year,precip_regional_mm
110,2010,219.86
111,2011,251.48
112,2012,244.82
113,2013,162.4
114,2014,244.92
115,2015,273.83
116,2016,169.8
117,2017,288.11
118,2018,180.94
119,2019,96.89


In [37]:
dataset_regional = pd.merge(oni_mjja, precip_regional, on="year")
dataset_regional_roni = pd.merge(roni_mjja, precip_regional, on="year")
dataset_regional_aao = pd.merge(aao_mjja, precip_regional, on="year")

p1 = dataset_regional[dataset_regional["year"] < 2000]
p2 = dataset_regional[dataset_regional["year"] >= 2000]

print("=== ONI vs precipitación REGIONAL (5 estaciones) ===")
print("1974-1999:", p1[["oni_mjja","precip_regional_mm"]].corr().iloc[0,1])
print("2000-2020:", p2[["oni_mjja","precip_regional_mm"]].corr().iloc[0,1])

p2_roni = dataset_regional_roni[dataset_regional_roni["year"] >= 2000]
print("\n2000-2020 con RONI:", p2_roni[["roni_mjja","precip_regional_mm"]].corr().iloc[0,1])

p2_aao = dataset_regional_aao[dataset_regional_aao["year"] >= 2000]
print("2000-2020 con AAO:", p2_aao[["aao_mjja","precip_regional_mm"]].corr().iloc[0,1])

=== ONI vs precipitación REGIONAL (5 estaciones) ===
1974-1999: 0.4742424504901564
2000-2020: -0.03082992236513306

2000-2020 con RONI: 0.1808696252884409
2000-2020 con AAO: -0.2767903601542688
